# Token and Style Embeddings

Notebook for embedding layer implementation

## 1 Imports

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from collections import Counter
fr

print('True')

True


## 2. Reload CSV

In [2]:
path = '/kaggle/input/datasets/manjushwarkhairkar/cleaned-fashion-tech-myntra-dataset-with-features/fashion_preprocessed.csv'
df = pd.read_csv(path)
df = df.dropna(subset=['proc_img_path', 'proc_sketch_path']).reset_index(drop=True)
print(f"Loaded {len(df)} rows")
print(df[['name', 'colour']].head(5))

Loaded 14311 rows
                                                name     colour
0  Khushal K Women Black Ethnic Motifs Printed Ku...      Black
1  InWeave Women Orange Solid Kurta with Palazzos...     Orange
2  Anubhutee Women Navy Blue Ethnic Motifs Embroi...  Navy Blue
3  Nayo Women Red Floral Printed Kurta With Trous...        Red
4   AHIKA Women Black & Green Printed Straight Kurta      Black


## 3. Building Color Vocabulary

In [12]:
def build_vocab(series, max_vocab=200):
    # count all unique values
    counter = Counter(series.str.lower().str.strip())
    # most common first, reserve index 0 for <PAD>, last for <UNK>
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, _ in counter.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    return vocab

colour_vocab = build_vocab(df['colour'])
print(f"Colour vocabulary size : {len(colour_vocab)}")
print(f"Sample entries         : ")

for i in range(0,51):
    print(f"{dict(list(colour_vocab.items())[i:i+1])}")

Colour vocabulary size : 52
Sample entries         : 
{'<PAD>': 0}
{'<UNK>': 1}
{'black': 2}
{'blue': 3}
{'pink': 4}
{'green': 5}
{'navy blue': 6}
{'white': 7}
{'red': 8}
{'grey': 9}
{'maroon': 10}
{'yellow': 11}
{'beige': 12}
{'mustard': 13}
{'off white': 14}
{'peach': 15}
{'purple': 16}
{'orange': 17}
{'olive': 18}
{'brown': 19}
{'teal': 20}
{'cream': 21}
{'multi': 22}
{'burgundy': 23}
{'sea green': 24}
{'turquoise blue': 25}
{'rust': 26}
{'lavender': 27}
{'gold': 28}
{'magenta': 29}
{'charcoal': 30}
{'coral': 31}
{'fuchsia': 32}
{'mauve': 33}
{'lime green': 34}
{'rose': 35}
{'grey melange': 36}
{'khaki': 37}
{'coffee brown': 38}
{'taupe': 39}
{'fluorescent green': 40}
{'violet': 41}
{'silver': 42}
{'tan': 43}
{'camel brown': 44}
{'nude': 45}
{'copper': 46}
{'unknown': 47}
{'assorted': 48}
{'bronze': 49}
{'champagne': 50}


## 4. Building category vocabulary from product names

In [13]:
def extract_category(name):
    # common fashion categories to look for
    categories = [
        'kurta', 'dress', 'top', 'palazzo', 'dupatta', 'saree',
        'skirt', 'jeans', 'shirt', 'jacket', 'shorts', 'leggings',
        'kurti', 'suit', 'gown', 'cardigan', 'sweater', 'hoodie',
        'trousers', 'tee', 'tank', 'blouse', 'shrug', 'tunic'
    ]
    name_lower = name.lower()
    for cat in categories:
        if cat in name_lower:
            return cat
    return 'other'

df['category'] = df['name'].apply(extract_category)
print(df['category'].value_counts().head(10))

category_vocab = build_vocab(df['category'], max_vocab=50)
print(f"\nCategory vocabulary size: {len(category_vocab)}")
print(f"Categories found       : {list(category_vocab.keys())}")


category
top         1671
dupatta     1649
skirt       1037
jeans        995
other        985
saree        980
jacket       930
kurta        926
dress        870
trousers     852
Name: count, dtype: int64

Category vocabulary size: 27
Categories found       : ['<PAD>', '<UNK>', 'top', 'dupatta', 'skirt', 'jeans', 'other', 'saree', 'jacket', 'kurta', 'dress', 'trousers', 'suit', 'shirt', 'shorts', 'palazzo', 'shrug', 'kurti', 'tunic', 'cardigan', 'sweater', 'blouse', 'hoodie', 'tank', 'gown', 'leggings', 'tee']


## 5. Columns Indexing

In [24]:
def text_to_idx(series, vocab):
    return series.str.lower().str.strip().map(
        lambda x: vocab.get(x, vocab['<UNK>'])
    ).values

df['colour_idx']   = text_to_idx(df['colour'],   colour_vocab)
df['category_idx'] = text_to_idx(df['category'], category_vocab)

print("Columns now:", df.columns.tolist())
print("Colour indices sample  :", df['colour_idx'][:5].tolist())
print("Category indices sample:", df['category_idx'][:5].tolist())

Columns now: ['p_id', 'name', 'price', 'colour', 'brand', 'img', 'ratingCount', 'avg_rating', 'description', 'p_attributes', 'tok_len', 'attr_keys', 'img_path', 'proc_img_path', 'proc_sketch_path', 'category', 'colour_idx', 'category_idx']
Colour indices sample  : [2, 17, 6, 8, 2]
Category indices sample: [9, 9, 9, 9, 9]


## 6. Style Embedding class


In [31]:
class StyleEmbedding(nn.Module):

    def __init__(self,
                 colour_vocab_size,
                 category_vocab_size,
                 embed_dim=64):
        super().__init__()

        self.embed_dim = embed_dim

        self.colour_emb   = nn.Embedding(colour_vocab_size,   embed_dim, padding_idx=0)
        self.category_emb = nn.Embedding(category_vocab_size, embed_dim, padding_idx=0)

        self.fusion = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

    def forward(self, colour_idx, category_idx):
        c = self.colour_emb(colour_idx)
        s = self.category_emb(category_idx)
        fused = torch.cat([c, s], dim=-1)
        style_vector = self.fusion(fused)
        return style_vector

style_embedder = StyleEmbedding(
    colour_vocab_size   = len(colour_vocab),
    category_vocab_size = len(category_vocab),
    embed_dim           = 64
)
print("StyleEmbedding ready!")
print('True')

StyleEmbedding ready!
True


## 8. Sanity Checking with real batch

In [32]:
# Cell 8 — Sanity check with real batch
batch_size = 32

colour_idx_batch   = torch.tensor(df['colour_idx'][:batch_size].tolist(),   dtype=torch.long)
category_idx_batch = torch.tensor(df['category_idx'][:batch_size].tolist(), dtype=torch.long)

with torch.no_grad():
    style_vec = style_embedder(colour_idx_batch, category_idx_batch)

print(f"Colour idx shape  : {colour_idx_batch.shape}")
print(f"Category idx shape: {category_idx_batch.shape}")
print(f"Style vector shape: {style_vec.shape}")
print(f"Expected          : torch.Size([32, 128])")
print(f"Match             : {style_vec.shape == torch.Size([32, 128])}")

Colour idx shape  : torch.Size([32])
Category idx shape: torch.Size([32])
Style vector shape: torch.Size([32, 128])
Expected          : torch.Size([32, 128])
Match             : True


## 9. Saving vocab and embedder

In [36]:
import json
import os

# create directory if it doesn't exist
os.makedirs('/kaggle/working/vocab_and_category', exist_ok=True)

# save vocabularies as JSON
with open('/kaggle/working/vocab_and_category/colour_vocab.json', 'w') as f:
    json.dump(colour_vocab, f)
with open('/kaggle/working/vocab_and_category/category_vocab.json', 'w') as f:
    json.dump(category_vocab, f)

# save updated df with colour_idx and category_idx columns
df.to_csv('/kaggle/working/vocab_and_category/fashion_with_indices.csv', index=False)

# save embedder weights
torch.save(style_embedder.state_dict(), '/kaggle/working/vocab_and_category/style_embedder.pt')

print("Saved:")
print("  colour_vocab.json")
print("  category_vocab.json")
print("  fashion_with_indices.csv")
print("  style_embedder.pt")

Saved:
  colour_vocab.json
  category_vocab.json
  fashion_with_indices.csv
  style_embedder.pt
